In [3]:
# =========================
# Project paths
# =========================

PROJECT_ROOT = Path.home() / "Documents" / "recommender_project" / "trial three"

RAW_DIR = PROJECT_ROOT / "data_raw"
PROCESSED_DIR = PROJECT_ROOT / "data_preprocessed"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"

# Create reports folder if it does not exist
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR:", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("MODELS_DIR:", MODELS_DIR)
print("REPORTS_DIR:", REPORTS_DIR)

PROJECT_ROOT: C:\Users\pc\Documents\recommender_project\trial two
RAW_DIR: C:\Users\pc\Documents\recommender_project\trial two\data_raw
PROCESSED_DIR: C:\Users\pc\Documents\recommender_project\trial two\data_preprocessed
MODELS_DIR: C:\Users\pc\Documents\recommender_project\trial two\models
REPORTS_DIR: C:\Users\pc\Documents\recommender_project\trial two\reports


In [4]:
# =========================
# Raw file paths
# =========================

events_path = RAW_DIR / "events.csv"
category_tree_path = RAW_DIR / "category_tree.csv"
item_properties_part1_path = RAW_DIR / "item_properties_part1.csv"
item_properties_part2_path = RAW_DIR / "item_properties_part2.csv"

raw_files = {
    "events": events_path,
    "category_tree": category_tree_path,
    "item_properties_part1": item_properties_part1_path,
    "item_properties_part2": item_properties_part2_path
}

for name, path in raw_files.items():
    print(f"{name}: {path}")

events: C:\Users\pc\Documents\recommender_project\trial two\data_raw\events.csv
category_tree: C:\Users\pc\Documents\recommender_project\trial two\data_raw\category_tree.csv
item_properties_part1: C:\Users\pc\Documents\recommender_project\trial two\data_raw\item_properties_part1.csv
item_properties_part2: C:\Users\pc\Documents\recommender_project\trial two\data_raw\item_properties_part2.csv


In [7]:
# =========================
# Load events.csv
# =========================

events = pd.read_csv(
    events_path,
    dtype={
        "visitorid": "int64",
        "event": "category",
        "itemid": "int64",
        "transactionid": "float64"
    }
)

print("events loaded successfully.")
print("events shape:", events.shape)

events.head()

events loaded successfully.
events shape: (2756101, 5)


,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN


In [10]:
# =========================
# Load category_tree.csv
# =========================

category_tree = pd.read_csv(category_tree_path)

print("category_tree loaded successfully.")
print("category_tree shape:", category_tree.shape)

category_tree.head()

category_tree loaded successfully.
category_tree shape: (1669, 2)


,categoryid,parentid
0,1016,213.0
1,809,169.0
2,570,9.0
3,1691,885.0
4,536,1691.0


In [11]:
# =========================
# Preview item property files
# =========================

item_properties_part1_preview = pd.read_csv(item_properties_part1_path, nrows=5)
item_properties_part2_preview = pd.read_csv(item_properties_part2_path, nrows=5)

print("item_properties_part1 preview:")
display(item_properties_part1_preview)

print("item_properties_part2 preview:")
display(item_properties_part2_preview)

item_properties_part1 preview:


,timestamp,itemid,property,value
0,1435460400000,460429,categoryid,1338
1,1441508400000,206783,888,1116713 960601 n277.200
2,1439089200000,395014,400,n552.000 639502 n720.000 424566
3,1431226800000,59481,790,n15360.000
4,1431831600000,156781,917,828513


item_properties_part2 preview:


,timestamp,itemid,property,value
0,1433041200000,183478,561,769062
1,1439694000000,132256,976,n26.400 1135780
2,1435460400000,420307,921,1149317 1257525
3,1431831600000,403324,917,1204143
4,1435460400000,230701,521,769062


In [12]:
# =========================
# Quick raw data overview
# =========================

print("===== EVENTS OVERVIEW =====")
print("Shape:", events.shape)
print("Columns:", events.columns.tolist())
print("\nData types:")
print(events.dtypes)

print("\nMissing values:")
print(events.isna().sum())

print("\nEvent counts:")
print(events["event"].value_counts())

print("\nUnique users:", events["visitorid"].nunique())
print("Unique items:", events["itemid"].nunique())

print("\n===== CATEGORY TREE OVERVIEW =====")
print("Shape:", category_tree.shape)
print("Columns:", category_tree.columns.tolist())
print("\nMissing values:")
print(category_tree.isna().sum())

===== EVENTS OVERVIEW =====
Shape: (2756101, 5)
Columns: ['timestamp', 'visitorid', 'event', 'itemid', 'transactionid']

Data types:
timestamp           int64
visitorid           int64
event            category
itemid              int64
transactionid     float64
dtype: object

Missing values:
timestamp              0
visitorid              0
event                  0
itemid                 0
transactionid    2733644
dtype: int64

Event counts:
event
view           2664312
addtocart        69332
transaction      22457
Name: count, dtype: int64

Unique users: 1407580
Unique items: 235061

===== CATEGORY TREE OVERVIEW =====
Shape: (1669, 2)
Columns: ['categoryid', 'parentid']

Missing values:
categoryid     0
parentid      25
dtype: int64


In [13]:
# =========================
# Step 2.1 - Clean events: timestamp + duplicates

events_clean = events.copy()

print("Original events shape:", events_clean.shape)

# Convert timestamp from milliseconds to readable datetime
events_clean["timestamp"] = pd.to_datetime(events_clean["timestamp"], unit="ms")

# Check duplicate rows before removing
duplicate_count = events_clean.duplicated().sum()
print("Duplicate rows found:", duplicate_count)



Original events shape: (2756101, 5)
Duplicate rows found: 460


In [14]:
# Remove exact duplicate rows
events_clean = events_clean.drop_duplicates().reset_index(drop=True)

print("Shape after removing duplicates:", events_clean.shape)
print("Date range:")
print("Start:", events_clean["timestamp"].min())
print("End:", events_clean["timestamp"].max())

events_clean.head()

Shape after removing duplicates: (2755641, 5)
Date range:
Start: 2015-05-03 03:00:04.384000
End: 2015-09-18 02:59:47.788000


,timestamp,visitorid,event,itemid,transactionid
0,2015-06-02 05:02:12.117,257597,view,355908,NaN
1,2015-06-02 05:50:14.164,992329,view,248676,NaN
2,2015-06-02 05:13:19.827,111016,view,318965,NaN
3,2015-06-02 05:12:35.914,483717,view,253185,NaN
4,2015-06-02 05:02:17.106,951259,view,367447,NaN


In [15]:
# =========================
# Step 2.2 - Validate event types
# =========================

valid_events = ["view", "addtocart", "transaction"]

print("Unique event types before filtering:")
print(events_clean["event"].value_counts())

# Keep only valid Retailrocket event types
events_clean = events_clean[events_clean["event"].isin(valid_events)].copy()

print("\nUnique event types after filtering:")
print(events_clean["event"].value_counts())

print("\nShape after event validation:", events_clean.shape)

Unique event types before filtering:
event
view           2664218
addtocart        68966
transaction      22457
Name: count, dtype: int64

Unique event types after filtering:
event
view           2664218
addtocart        68966
transaction      22457
Name: count, dtype: int64

Shape after event validation: (2755641, 5)


In [16]:
# =========================
# Step 2.3 - Missing values check
# =========================

print("Missing values after cleaning:")
print(events_clean.isna().sum())

print("\nMissing transactionid by event type:")
print(
    events_clean
    .groupby("event")["transactionid"]
    .apply(lambda x: x.isna().sum())
)

print("\nNon-missing transactionid by event type:")
print(
    events_clean
    .groupby("event")["transactionid"]
    .apply(lambda x: x.notna().sum())
)

Missing values after cleaning:
timestamp              0
visitorid              0
event                  0
itemid                 0
transactionid    2733184
dtype: int64

Missing transactionid by event type:
event
addtocart        68966
transaction          0
view           2664218
Name: transactionid, dtype: int64

Non-missing transactionid by event type:
event
addtocart          0
transaction    22457
view               0
Name: transactionid, dtype: int64


In [17]:
# =========================
# Step 2.4 - Drop missing essential values only
# =========================

before_rows = len(events_clean)

essential_columns = ["timestamp", "visitorid", "event", "itemid"]

events_clean = events_clean.dropna(subset=essential_columns).copy()

after_rows = len(events_clean)

print("Rows before:", before_rows)
print("Rows after:", after_rows)
print("Rows removed:", before_rows - after_rows)

print("\nFinal missing values:")
print(events_clean.isna().sum())

Rows before: 2755641
Rows after: 2755641
Rows removed: 0

Final missing values:
timestamp              0
visitorid              0
event                  0
itemid                 0
transactionid    2733184
dtype: int64


In [18]:
# =========================
# Step 2.5 - Sort events chronologically per user
# =========================

events_clean = events_clean.sort_values(
    ["visitorid", "timestamp"]
).reset_index(drop=True)

print("Events sorted by visitorid and timestamp.")
print("Final cleaned events shape:", events_clean.shape)

events_clean.head(10)

Events sorted by visitorid and timestamp.
Final cleaned events shape: (2755641, 5)


,timestamp,visitorid,event,itemid,transactionid
0,2015-09-11 20:49:49.439,0,view,285930,NaN
1,2015-09-11 20:52:39.591,0,view,357564,NaN
2,2015-09-11 20:55:17.175,0,view,67045,NaN
3,2015-08-13 17:46:06.444,1,view,72028,NaN
4,2015-08-07 17:51:44.567,2,view,325215,NaN
5,2015-08-07 17:53:33.790,2,view,325215,NaN
6,2015-08-07 17:56:52.664,2,view,259884,NaN
7,2015-08-07 18:01:08.920,2,view,216305,NaN
8,2015-08-07 18:08:25.669,2,view,342816,NaN
9,2015-08-07 18:17:24.375,2,view,342816,NaN


In [19]:
# =========================
# Step 2.6 - Save cleaned events
# =========================

clean_events_path = PROCESSED_DIR / "events_clean.parquet"

events_clean.to_parquet(clean_events_path, index=False)

print("Saved cleaned events to:")
print(clean_events_path)

print("\nCleaned events summary:")
print("Rows:", len(events_clean))
print("Unique users:", events_clean["visitorid"].nunique())
print("Unique items:", events_clean["itemid"].nunique())
print("Date range:", events_clean["timestamp"].min(), "to", events_clean["timestamp"].max())
print("\nEvent counts:")
print(events_clean["event"].value_counts())

Saved cleaned events to:
C:\Users\pc\Documents\recommender_project\trial two\data_preprocessed\events_clean.parquet

Cleaned events summary:
Rows: 2755641
Unique users: 1407580
Unique items: 235061
Date range: 2015-05-03 03:00:04.384000 to 2015-09-18 02:59:47.788000

Event counts:
event
view           2664218
addtocart        68966
transaction      22457
Name: count, dtype: int64


In [20]:
# =========================
# Step 3.1 - Create event weights
# =========================

event_weights = {
    "view": 1.0,
    "addtocart": 3.0,
    "transaction": 10.0
}

events_clean["event_weight"] = events_clean["event"].map(event_weights).astype("float32")

print("Event weights added.")
print(events_clean[["event", "event_weight"]].drop_duplicates().sort_values("event_weight"))

print("\nWeighted event summary:")
print(
    events_clean
    .groupby("event")["event_weight"]
    .agg(["count", "mean", "sum"])
)

Event weights added.
           event  event_weight
0           view           1.0
15     addtocart           3.0
363  transaction          10.0

Weighted event summary:
               count  mean        sum
event                                
addtocart      68966   3.0   206898.0
transaction    22457  10.0   224570.0
view         2664218   1.0  2664218.0


In [21]:
# =========================
# Step 3.2 - Create recency weight
# =========================

# Most recent timestamp in the dataset
max_timestamp = events_clean["timestamp"].max()

# How many days ago each event happened
events_clean["days_ago"] = (
    max_timestamp - events_clean["timestamp"]
).dt.total_seconds() / (60 * 60 * 24)

# Half-life controls how fast old events lose importance
HALF_LIFE_DAYS = 30

events_clean["recency_weight"] = np.exp(
    -np.log(2) * events_clean["days_ago"] / HALF_LIFE_DAYS
).astype("float32")

# Final interaction strength
events_clean["final_weight"] = (
    events_clean["event_weight"] * events_clean["recency_weight"]
).astype("float32")

print("Recency weights added.")
print("Max timestamp:", max_timestamp)
print("Half-life days:", HALF_LIFE_DAYS)

events_clean[[
    "visitorid",
    "itemid",
    "event",
    "timestamp",
    "event_weight",
    "days_ago",
    "recency_weight",
    "final_weight"
]].head(10)

Recency weights added.
Max timestamp: 2015-09-18 02:59:47.788000
Half-life days: 30


,visitorid,itemid,event,timestamp,event_weight,days_ago,recency_weight,final_weight
0,0,285930,view,2015-09-11 20:49:49.439,1.0,6.256925,0.865398,0.865398
1,0,357564,view,2015-09-11 20:52:39.591,1.0,6.254956,0.865437,0.865437
2,0,67045,view,2015-09-11 20:55:17.175,1.0,6.253132,0.865474,0.865474
3,1,72028,view,2015-08-13 17:46:06.444,1.0,35.384506,0.441510,0.441510
4,2,325215,view,2015-08-07 17:51:44.567,1.0,41.380593,0.384391,0.384391
5,2,325215,view,2015-08-07 17:53:33.790,1.0,41.379329,0.384402,0.384402
6,2,259884,view,2015-08-07 17:56:52.664,1.0,41.377027,0.384423,0.384423
7,2,216305,view,2015-08-07 18:01:08.920,1.0,41.374061,0.384449,0.384449
8,2,342816,view,2015-08-07 18:08:25.669,1.0,41.369006,0.384494,0.384494
9,2,342816,view,2015-08-07 18:17:24.375,1.0,41.362771,0.384549,0.384549


In [22]:
# =========================
# Step 3.3 - Sanity check weights
# =========================

print("Final weight summary:")
print(events_clean["final_weight"].describe())

print("\nRecency weight summary:")
print(events_clean["recency_weight"].describe())

print("\nFinal weight by event type:")
print(
    events_clean
    .groupby("event")["final_weight"]
    .agg(["count", "min", "mean", "max", "sum"])
    .sort_values("sum", ascending=False)
)

Final weight summary:
count    2.755641e+06
mean     3.210331e-01
std      4.281721e-01
min      4.123488e-02
25%      9.127706e-02
50%      2.021403e-01
75%      4.320731e-01
max      9.997337e+00
Name: final_weight, dtype: float64

Recency weight summary:
count    2.755641e+06
mean     2.854311e-01
std      2.470135e-01
min      4.123481e-02
25%      8.850439e-02
50%      1.960833e-01
75%      4.115330e-01
max      1.000000e+00
Name: recency_weight, dtype: float64

Final weight by event type:
               count       min      mean       max            sum
event                                                            
view         2664218  0.041235  0.285270  1.000000  760022.750000
transaction    22457  0.412529  2.866411  9.997337   64371.003906
addtocart      68966  0.123704  0.873735  2.999568   60258.019531


In [23]:
# =========================
# Step 3.4 - Compare event count and weighted strength
# =========================

event_importance_summary = (
    events_clean
    .groupby("event")
    .agg(
        event_count=("event", "count"),
        total_event_weight=("event_weight", "sum"),
        total_final_weight=("final_weight", "sum"),
        avg_final_weight=("final_weight", "mean")
    )
    .sort_values("total_final_weight", ascending=False)
)

event_importance_summary["event_count_percentage"] = (
    event_importance_summary["event_count"] / event_importance_summary["event_count"].sum()
) * 100

event_importance_summary["final_weight_percentage"] = (
    event_importance_summary["total_final_weight"] / event_importance_summary["total_final_weight"].sum()
) * 100

event_importance_summary

,event_count,total_event_weight,total_final_weight,avg_final_weight,event_count_percentage,final_weight_percentage
event,,,,,,
view,2664218,2664218.0,760022.750000,0.285270,96.682333,85.912086
transaction,22457,224570.0,64371.003906,2.866411,0.814947,7.276423
addtocart,68966,206898.0,60258.019531,0.873735,2.502721,6.811496


In [24]:
# =========================
# Step 3.5 - Save weighted event-level data
# =========================

weighted_events_path = PROCESSED_DIR / "events_weighted.parquet"

events_clean.to_parquet(weighted_events_path, index=False)

print("Saved weighted events to:")
print(weighted_events_path)

print("\nWeighted events shape:", events_clean.shape)
print("Columns:")
print(events_clean.columns.tolist())

Saved weighted events to:
C:\Users\pc\Documents\recommender_project\trial two\data_preprocessed\events_weighted.parquet

Weighted events shape: (2755641, 9)
Columns:
['timestamp', 'visitorid', 'event', 'itemid', 'transactionid', 'event_weight', 'days_ago', 'recency_weight', 'final_weight']


In [25]:
# =========================
# Step 4.1 - Aggregate user-item interactions
# =========================

interactions = (
    events_clean
    .groupby(["visitorid", "itemid"], as_index=False)
    .agg(
        interaction_strength=("final_weight", "sum"),
        raw_event_strength=("event_weight", "sum"),
        num_events=("event", "count"),
        max_event_weight=("event_weight", "max"),
        first_interaction=("timestamp", "min"),
        last_interaction=("timestamp", "max")
    )
)

print("Aggregated user-item interactions created.")
print("Shape:", interactions.shape)

interactions.head(10)

Aggregated user-item interactions created.
Shape: (2145179, 8)


,visitorid,itemid,interaction_strength,raw_event_strength,num_events,max_event_weight,first_interaction,last_interaction
0,0,67045,0.865474,1.0,1,1.0,2015-09-11 20:55:17.175,2015-09-11 20:55:17.175
1,0,285930,0.865398,1.0,1,1.0,2015-09-11 20:49:49.439,2015-09-11 20:49:49.439
2,0,357564,0.865437,1.0,1,1.0,2015-09-11 20:52:39.591,2015-09-11 20:52:39.591
3,1,72028,0.441510,1.0,1,1.0,2015-08-13 17:46:06.444,2015-08-13 17:46:06.444
4,2,216305,0.769001,2.0,2,1.0,2015-08-07 18:01:08.920,2015-08-07 18:17:43.170
5,2,259884,0.384423,1.0,1,1.0,2015-08-07 17:56:52.664,2015-08-07 17:56:52.664
6,2,325215,1.153365,3.0,3,1.0,2015-08-07 17:51:44.567,2015-08-07 18:20:57.845
7,2,342816,0.769043,2.0,2,1.0,2015-08-07 18:08:25.669,2015-08-07 18:17:24.375
8,3,385090,0.331207,1.0,1,1.0,2015-08-01 07:10:35.296,2015-08-01 07:10:35.296
9,4,177677,0.949718,1.0,1,1.0,2015-09-15 21:24:27.167,2015-09-15 21:24:27.167


In [26]:
# =========================
# Step 4.2 - Add high-intent flag
# =========================

interactions["is_high_intent"] = interactions["max_event_weight"] >= 3.0

print("High-intent flag added.")

print(interactions["is_high_intent"].value_counts())

print("\nHigh-intent percentage:")
print(interactions["is_high_intent"].value_counts(normalize=True) * 100)

interactions.head()

High-intent flag added.
is_high_intent
False    2080929
True       64250
Name: count, dtype: int64

High-intent percentage:
is_high_intent
False    97.004912
True      2.995088
Name: proportion, dtype: float64


,visitorid,itemid,interaction_strength,raw_event_strength,num_events,max_event_weight,first_interaction,last_interaction,is_high_intent
0,0,67045,0.865474,1.0,1,1.0,2015-09-11 20:55:17.175,2015-09-11 20:55:17.175,False
1,0,285930,0.865398,1.0,1,1.0,2015-09-11 20:49:49.439,2015-09-11 20:49:49.439,False
2,0,357564,0.865437,1.0,1,1.0,2015-09-11 20:52:39.591,2015-09-11 20:52:39.591,False
3,1,72028,0.441510,1.0,1,1.0,2015-08-13 17:46:06.444,2015-08-13 17:46:06.444,False
4,2,216305,0.769001,2.0,2,1.0,2015-08-07 18:01:08.920,2015-08-07 18:17:43.170,False


In [27]:
# =========================
# Step 4.3 - Check aggregation effect
# =========================

event_rows = len(events_clean)
unique_user_item_pairs = len(interactions)

reduction = 1 - (unique_user_item_pairs / event_rows)

print("===== AGGREGATION SUMMARY =====")
print(f"Original event rows: {event_rows:,}")
print(f"Unique user-item pairs: {unique_user_item_pairs:,}")
print(f"Rows reduced by: {reduction * 100:.2f}%")

print("\nInteraction strength summary:")
print(interactions["interaction_strength"].describe())

print("\nNumber of events per user-item pair:")
print(interactions["num_events"].describe())

===== AGGREGATION SUMMARY =====
Original event rows: 2,755,641
Unique user-item pairs: 2,145,179
Rows reduced by: 22.15%

Interaction strength summary:
count    2.145179e+06
mean     4.123906e-01
std      8.021846e-01
min      4.123488e-02
25%      1.019250e-01
50%      2.279505e-01
75%      4.997445e-01
max      1.285428e+02
Name: interaction_strength, dtype: float64

Number of events per user-item pair:
count    2.145179e+06
mean     1.284574e+00
std      1.231457e+00
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      3.080000e+02
Name: num_events, dtype: float64


In [28]:
# =========================
# Step 4.4 - Check sparsity from aggregated interactions
# =========================

num_users = interactions["visitorid"].nunique()
num_items = interactions["itemid"].nunique()

observed_interactions = interactions[["visitorid", "itemid"]].drop_duplicates().shape[0]
total_possible_interactions = num_users * num_items

density = observed_interactions / total_possible_interactions
sparsity = 1 - density

print("===== USER-ITEM SPARSITY BEFORE FILTERING =====")
print(f"Number of users: {num_users:,}")
print(f"Number of items: {num_items:,}")
print(f"Observed user-item pairs: {observed_interactions:,}")
print(f"Total possible user-item pairs: {total_possible_interactions:,}")
print(f"Density: {density:.10f}")
print(f"Density percentage: {density * 100:.6f}%")
print(f"Sparsity: {sparsity:.10f}")
print(f"Sparsity percentage: {sparsity * 100:.6f}%")

===== USER-ITEM SPARSITY BEFORE FILTERING =====
Number of users: 1,407,580
Number of items: 235,061
Observed user-item pairs: 2,145,179
Total possible user-item pairs: 330,867,162,380
Density: 0.0000064835
Density percentage: 0.000648%
Sparsity: 0.9999935165
Sparsity percentage: 99.999352%


In [29]:
# =========================
# Step 4.5 - Save aggregated interactions
# =========================

interactions_path = PROCESSED_DIR / "interactions_aggregated.parquet"

interactions.to_parquet(interactions_path, index=False)

print("Saved aggregated interactions to:")
print(interactions_path)

print("\nAggregated interactions shape:", interactions.shape)
print("Columns:")
print(interactions.columns.tolist())

Saved aggregated interactions to:
C:\Users\pc\Documents\recommender_project\trial two\data_preprocessed\interactions_aggregated.parquet

Aggregated interactions shape: (2145179, 9)
Columns:
['visitorid', 'itemid', 'interaction_strength', 'raw_event_strength', 'num_events', 'max_event_weight', 'first_interaction', 'last_interaction', 'is_high_intent']


In [30]:
# =========================
# Step 5.1 - Check user/item interaction distribution
# =========================

user_interaction_counts = interactions.groupby("visitorid")["itemid"].nunique()
item_interaction_counts = interactions.groupby("itemid")["visitorid"].nunique()

print("===== USER INTERACTION COUNTS =====")
print(user_interaction_counts.describe())

print("\nUsers with only 1 item:", (user_interaction_counts == 1).sum())
print("Users with fewer than 3 items:", (user_interaction_counts < 3).sum())
print("Users with fewer than 5 items:", (user_interaction_counts < 5).sum())

print("\n===== ITEM INTERACTION COUNTS =====")
print(item_interaction_counts.describe())

print("\nItems with only 1 user:", (item_interaction_counts == 1).sum())
print("Items with fewer than 3 users:", (item_interaction_counts < 3).sum())
print("Items with fewer than 5 users:", (item_interaction_counts < 5).sum())

===== USER INTERACTION COUNTS =====
count    1.407580e+06
mean     1.524019e+00
std      7.143724e+00
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      3.814000e+03
Name: itemid, dtype: float64

Users with only 1 item: 1119500
Users with fewer than 3 items: 1290051
Users with fewer than 5 items: 1367945

===== ITEM INTERACTION COUNTS =====
count    235061.000000
mean          9.126052
std          27.102543
min           1.000000
25%           1.000000
50%           2.000000
75%           7.000000
max        2912.000000
Name: visitorid, dtype: float64

Items with only 1 user: 81347
Items with fewer than 3 users: 117533
Items with fewer than 5 users: 153156


In [32]:
# =========================
# Step 5.2 - Define iterative filtering function
# =========================

def iterative_filter_interactions(
    df,
    min_user_interactions=5,
    min_item_interactions=5,
    user_col="visitorid",
    item_col="itemid"
):
    """
    Iteratively filter sparse users and sparse items.

    Keeps:
    - users with at least min_user_interactions unique items
    - items with at least min_item_interactions unique users
    """

    filtered = df.copy()
    iteration = 0

    while True:
        iteration += 1
        rows_before = len(filtered)
        users_before = filtered[user_col].nunique()
        items_before = filtered[item_col].nunique()

        # Count unique items per user
        user_counts = filtered.groupby(user_col)[item_col].nunique()

        # Count unique users per item
        item_counts = filtered.groupby(item_col)[user_col].nunique()

        valid_users = user_counts[user_counts >= min_user_interactions].index
        valid_items = item_counts[item_counts >= min_item_interactions].index

        filtered = filtered[
            filtered[user_col].isin(valid_users)
            & filtered[item_col].isin(valid_items)
        ].copy()

        rows_after = len(filtered)
        users_after = filtered[user_col].nunique()
        items_after = filtered[item_col].nunique()

        print(f"Iteration {iteration}")
        print(f"Rows:  {rows_before:,} -> {rows_after:,}")
        print(f"Users: {users_before:,} -> {users_after:,}")
        print(f"Items: {items_before:,} -> {items_after:,}")
        print("-" * 50)

        if rows_after == rows_before:
            print("Filtering is stable. Done.")
            break

        if rows_after == 0:
            print("Warning: filtering removed everything. Use smaller thresholds.")
            break

    return filtered

In [33]:
# =========================
# Step 5.3 - Apply filtering
# =========================

MIN_USER_INTERACTIONS = 5
MIN_ITEM_INTERACTIONS = 5

interactions_filtered = iterative_filter_interactions(
    interactions,
    min_user_interactions=MIN_USER_INTERACTIONS,
    min_item_interactions=MIN_ITEM_INTERACTIONS
)

print("\n===== FILTERED DATA SUMMARY =====")
print("Filtered shape:", interactions_filtered.shape)
print("Unique users:", interactions_filtered["visitorid"].nunique())
print("Unique items:", interactions_filtered["itemid"].nunique())

interactions_filtered.head()

Iteration 1
Rows:  2,145,179 -> 381,434
Users: 1,407,580 -> 39,424
Items: 235,061 -> 56,188
--------------------------------------------------
Iteration 2
Rows:  381,434 -> 303,020
Users: 39,424 -> 34,065
Items: 56,188 -> 23,188
--------------------------------------------------
Iteration 3
Rows:  303,020 -> 272,337
Users: 34,065 -> 25,518
Items: 23,188 -> 22,044
--------------------------------------------------
Iteration 4
Rows:  272,337 -> 261,480
Users: 25,518 -> 25,020
Items: 22,044 -> 19,445
--------------------------------------------------
Iteration 5
Rows:  261,480 -> 255,398
Users: 25,020 -> 23,620
Items: 19,445 -> 19,230
--------------------------------------------------
Iteration 6
Rows:  255,398 -> 252,704
Users: 23,620 -> 23,502
Items: 19,230 -> 18,643
--------------------------------------------------
Iteration 7
Rows:  252,704 -> 251,091
Users: 23,502 -> 23,143
Items: 18,643 -> 18,588
--------------------------------------------------
Iteration 8
Rows:  251,091 -> 250,2

,visitorid,itemid,interaction_strength,raw_event_strength,num_events,max_event_weight,first_interaction,last_interaction,is_high_intent
89,64,118401,0.113501,1.0,1,1.0,2015-06-15 22:45:31.466,2015-06-15 22:45:31.466,False
90,64,141657,0.113404,1.0,1,1.0,2015-06-15 21:51:51.942,2015-06-15 21:51:51.942,False
91,64,160984,0.250381,2.0,2,1.0,2015-06-15 22:58:56.974,2015-06-24 01:07:08.344,False
92,64,224700,0.113511,1.0,1,1.0,2015-06-15 22:51:00.770,2015-06-15 22:51:00.770,False
93,64,233200,0.113491,1.0,1,1.0,2015-06-15 22:40:04.268,2015-06-15 22:40:04.268,False


In [34]:
# =========================
# Step 5.4 - Check sparsity after filtering
# =========================

num_users_filtered = interactions_filtered["visitorid"].nunique()
num_items_filtered = interactions_filtered["itemid"].nunique()

observed_interactions_filtered = interactions_filtered[["visitorid", "itemid"]].drop_duplicates().shape[0]
total_possible_interactions_filtered = num_users_filtered * num_items_filtered

density_filtered = observed_interactions_filtered / total_possible_interactions_filtered
sparsity_filtered = 1 - density_filtered

print("===== USER-ITEM SPARSITY AFTER FILTERING =====")
print(f"Number of users: {num_users_filtered:,}")
print(f"Number of items: {num_items_filtered:,}")
print(f"Observed user-item pairs: {observed_interactions_filtered:,}")
print(f"Total possible user-item pairs: {total_possible_interactions_filtered:,}")
print(f"Density: {density_filtered:.10f}")
print(f"Density percentage: {density_filtered * 100:.6f}%")
print(f"Sparsity: {sparsity_filtered:.10f}")
print(f"Sparsity percentage: {sparsity_filtered * 100:.6f}%")

===== USER-ITEM SPARSITY AFTER FILTERING =====
Number of users: 22,890
Number of items: 18,269
Observed user-item pairs: 248,891
Total possible user-item pairs: 418,177,410
Density: 0.0005951804
Density percentage: 0.059518%
Sparsity: 0.9994048196
Sparsity percentage: 99.940482%


In [35]:
# =========================
# Step 5.5 - Check high-intent after filtering
# =========================

print("High-intent counts after filtering:")
print(interactions_filtered["is_high_intent"].value_counts())

print("\nHigh-intent percentage after filtering:")
print(interactions_filtered["is_high_intent"].value_counts(normalize=True) * 100)

print("\nMax event weight distribution:")
print(interactions_filtered["max_event_weight"].value_counts().sort_index())

High-intent counts after filtering:
is_high_intent
False    228404
True      20487
Name: count, dtype: int64

High-intent percentage after filtering:
is_high_intent
False    91.768686
True      8.231314
Name: proportion, dtype: float64

Max event weight distribution:
max_event_weight
1.0     228404
3.0      11649
10.0      8838
Name: count, dtype: int64


In [36]:
# =========================
# Step 5.6 - Save filtered interactions
# =========================

filtered_interactions_path = PROCESSED_DIR / "interactions_filtered.parquet"

interactions_filtered.to_parquet(filtered_interactions_path, index=False)

print("Saved filtered interactions to:")
print(filtered_interactions_path)

print("\nFiltered interactions shape:", interactions_filtered.shape)
print("Columns:")
print(interactions_filtered.columns.tolist())

Saved filtered interactions to:
C:\Users\pc\Documents\recommender_project\trial two\data_preprocessed\interactions_filtered.parquet

Filtered interactions shape: (248891, 9)
Columns:
['visitorid', 'itemid', 'interaction_strength', 'raw_event_strength', 'num_events', 'max_event_weight', 'first_interaction', 'last_interaction', 'is_high_intent']


In [37]:
# =========================
# Step 6.1 - Sort filtered interactions
# =========================

interactions_filtered = interactions_filtered.sort_values(
    ["visitorid", "last_interaction"]
).reset_index(drop=True)

print("Filtered interactions sorted by visitorid and last_interaction.")
print("Shape:", interactions_filtered.shape)

interactions_filtered.head(10)

Filtered interactions sorted by visitorid and last_interaction.
Shape: (248891, 9)


,visitorid,itemid,interaction_strength,raw_event_strength,num_events,max_event_weight,first_interaction,last_interaction,is_high_intent
0,64,141657,0.113404,1.0,1,1.0,2015-06-15 21:51:51.942,2015-06-15 21:51:51.942,False
1,64,233200,0.113491,1.0,1,1.0,2015-06-15 22:40:04.268,2015-06-15 22:40:04.268,False
2,64,118401,0.113501,1.0,1,1.0,2015-06-15 22:45:31.466,2015-06-15 22:45:31.466,False
3,64,224700,0.113511,1.0,1,1.0,2015-06-15 22:51:00.770,2015-06-15 22:51:00.770,False
4,64,160984,0.250381,2.0,2,1.0,2015-06-15 22:58:56.974,2015-06-24 01:07:08.344,False
5,155,123027,0.435783,1.0,1,1.0,2015-08-13 04:12:28.331,2015-08-13 04:12:28.331,False
6,155,134620,0.871564,2.0,2,1.0,2015-08-13 04:11:43.687,2015-08-13 04:12:56.509,False
7,155,50928,0.583315,1.0,1,1.0,2015-08-25 19:05:12.462,2015-08-25 19:05:12.462,False
8,155,373637,0.583320,1.0,1,1.0,2015-08-25 19:05:40.511,2015-08-25 19:05:40.511,False
9,155,151670,0.585647,1.0,1,1.0,2015-08-25 23:13:51.465,2015-08-25 23:13:51.465,False


In [38]:
# =========================
# Step 6.2 - Define per-user chronological split function
# =========================

def chronological_user_split(
    user_df,
    train_ratio=0.70,
    validation_ratio=0.15,
    test_ratio=0.15
):
    """
    Split one user's interactions chronologically.

    Oldest interactions go to train.
    Middle interactions go to validation.
    Latest interactions go to test.
    """
    
    user_df = user_df.sort_values("last_interaction").copy()
    n = len(user_df)

    # Safety case
    if n < 3:
        user_df["split"] = "train"
        return user_df

    # At least one validation and one test interaction when possible
    n_test = max(1, int(np.ceil(n * test_ratio)))
    n_validation = max(1, int(np.ceil(n * validation_ratio)))

    # Make sure train is not empty
    if n - n_validation - n_test < 1:
        n_test = 1
        n_validation = 1 if n >= 3 else 0

    n_train = n - n_validation - n_test

    # Extra safety
    if n_train < 1:
        n_train = 1
        remaining = n - n_train
        n_validation = max(0, remaining // 2)
        n_test = remaining - n_validation

    split_labels = (
        ["train"] * n_train
        + ["validation"] * n_validation
        + ["test"] * n_test
    )

    user_df["split"] = split_labels

    return user_df

In [39]:
# =========================
# Step 6.3 - Apply split per user
# =========================

interactions_split = (
    interactions_filtered
    .groupby("visitorid", group_keys=False)
    .apply(
        chronological_user_split,
        train_ratio=0.70,
        validation_ratio=0.15,
        test_ratio=0.15
    )
    .reset_index(drop=True)
)

print("Split applied.")

print("\nSplit counts:")
print(interactions_split["split"].value_counts())

print("\nSplit percentages:")
print(interactions_split["split"].value_counts(normalize=True) * 100)

interactions_split.head(10)

Split applied.

Split counts:
split
train         155049
validation     46921
test           46921
Name: count, dtype: int64

Split percentages:
split
train         62.295945
validation    18.852028
test          18.852028
Name: proportion, dtype: float64


,visitorid,itemid,interaction_strength,raw_event_strength,num_events,max_event_weight,first_interaction,last_interaction,is_high_intent,split
0,64,141657,0.113404,1.0,1,1.0,2015-06-15 21:51:51.942,2015-06-15 21:51:51.942,False,train
1,64,233200,0.113491,1.0,1,1.0,2015-06-15 22:40:04.268,2015-06-15 22:40:04.268,False,train
2,64,118401,0.113501,1.0,1,1.0,2015-06-15 22:45:31.466,2015-06-15 22:45:31.466,False,train
3,64,224700,0.113511,1.0,1,1.0,2015-06-15 22:51:00.770,2015-06-15 22:51:00.770,False,validation
4,64,160984,0.250381,2.0,2,1.0,2015-06-15 22:58:56.974,2015-06-24 01:07:08.344,False,test
5,155,123027,0.435783,1.0,1,1.0,2015-08-13 04:12:28.331,2015-08-13 04:12:28.331,False,train
6,155,134620,0.871564,2.0,2,1.0,2015-08-13 04:11:43.687,2015-08-13 04:12:56.509,False,train
7,155,50928,0.583315,1.0,1,1.0,2015-08-25 19:05:12.462,2015-08-25 19:05:12.462,False,train
8,155,373637,0.583320,1.0,1,1.0,2015-08-25 19:05:40.511,2015-08-25 19:05:40.511,False,validation
9,155,151670,0.585647,1.0,1,1.0,2015-08-25 23:13:51.465,2015-08-25 23:13:51.465,False,test


In [40]:
# =========================
# Step 6.4 - Create train/validation/test DataFrames
# =========================

train_interactions = interactions_split[
    interactions_split["split"] == "train"
].copy()

validation_interactions = interactions_split[
    interactions_split["split"] == "validation"
].copy()

test_interactions = interactions_split[
    interactions_split["split"] == "test"
].copy()

print("Train shape:", train_interactions.shape)
print("Validation shape:", validation_interactions.shape)
print("Test shape:", test_interactions.shape)

print("\nUnique users:")
print("Train users:", train_interactions["visitorid"].nunique())
print("Validation users:", validation_interactions["visitorid"].nunique())
print("Test users:", test_interactions["visitorid"].nunique())

print("\nUnique items:")
print("Train items:", train_interactions["itemid"].nunique())
print("Validation items:", validation_interactions["itemid"].nunique())
print("Test items:", test_interactions["itemid"].nunique())

Train shape: (155049, 10)
Validation shape: (46921, 10)
Test shape: (46921, 10)

Unique users:
Train users: 22890
Validation users: 22890
Test users: 22890

Unique items:
Train items: 18224
Validation items: 15309
Test items: 15195


In [41]:
# =========================
# Step 6.5 - Check split time order
# =========================

split_time_summary = (
    interactions_split
    .groupby("split")["last_interaction"]
    .agg(["min", "max"])
)

print("Split time summary:")
print(split_time_summary)

# Per-user leakage check
train_max_time = (
    train_interactions
    .groupby("visitorid")["last_interaction"]
    .max()
    .rename("train_max_time")
)

validation_min_time = (
    validation_interactions
    .groupby("visitorid")["last_interaction"]
    .min()
    .rename("validation_min_time")
)

test_min_time = (
    test_interactions
    .groupby("visitorid")["last_interaction"]
    .min()
    .rename("test_min_time")
)

validation_check = pd.concat([train_max_time, validation_min_time], axis=1).dropna()
test_check = pd.concat([train_max_time, test_min_time], axis=1).dropna()

validation_leakage_count = (
    validation_check["validation_min_time"] < validation_check["train_max_time"]
).sum()

test_leakage_count = (
    test_check["test_min_time"] < test_check["train_max_time"]
).sum()

print("\nPer-user chronological leakage check:")
print("Validation leakage users:", validation_leakage_count)
print("Test leakage users:", test_leakage_count)

Split time summary:
                               min                     max
split                                                     
test       2015-05-03 03:10:53.528 2015-09-18 02:56:44.475
train      2015-05-03 03:00:54.461 2015-09-18 01:50:56.376
validation 2015-05-03 03:07:39.202 2015-09-18 02:33:41.561

Per-user chronological leakage check:
Validation leakage users: 0
Test leakage users: 0


In [42]:
# =========================
# Step 6.6 - Check cold-start users/items
# =========================

def check_cold_start(train_df, eval_df, eval_name):
    train_users = set(train_df["visitorid"].unique())
    train_items = set(train_df["itemid"].unique())

    eval_users = set(eval_df["visitorid"].unique())
    eval_items = set(eval_df["itemid"].unique())

    cold_users = eval_users - train_users
    cold_items = eval_items - train_items

    print(f"===== {eval_name.upper()} COLD-START CHECK =====")
    print(f"{eval_name} users:", len(eval_users))
    print("Cold-start users:", len(cold_users))
    print(f"{eval_name} items:", len(eval_items))
    print("Cold-start items:", len(cold_items))
    print("-" * 50)

    return cold_users, cold_items


validation_cold_users, validation_cold_items = check_cold_start(
    train_interactions,
    validation_interactions,
    "validation"
)

test_cold_users, test_cold_items = check_cold_start(
    train_interactions,
    test_interactions,
    "test"
)

===== VALIDATION COLD-START CHECK =====
validation users: 22890
Cold-start users: 0
validation items: 15309
Cold-start items: 39
--------------------------------------------------
===== TEST COLD-START CHECK =====
test users: 22890
Cold-start users: 0
test items: 15195
Cold-start items: 43
--------------------------------------------------


In [43]:
# =========================
# Step 6.7 - Keep only known users/items for CF evaluation
# =========================

train_users = set(train_interactions["visitorid"].unique())
train_items = set(train_interactions["itemid"].unique())

validation_interactions_known = validation_interactions[
    validation_interactions["visitorid"].isin(train_users)
    & validation_interactions["itemid"].isin(train_items)
].copy()

test_interactions_known = test_interactions[
    test_interactions["visitorid"].isin(train_users)
    & test_interactions["itemid"].isin(train_items)
].copy()

print("Validation before filtering:", validation_interactions.shape)
print("Validation after known-user/item filtering:", validation_interactions_known.shape)

print("\nTest before filtering:", test_interactions.shape)
print("Test after known-user/item filtering:", test_interactions_known.shape)

print("\nRows removed from validation:", len(validation_interactions) - len(validation_interactions_known))
print("Rows removed from test:", len(test_interactions) - len(test_interactions_known))

Validation before filtering: (46921, 10)
Validation after known-user/item filtering: (46825, 10)

Test before filtering: (46921, 10)
Test after known-user/item filtering: (46763, 10)

Rows removed from validation: 96
Rows removed from test: 158


In [44]:
# =========================
# Step 6.8 - Save train/validation/test files
# =========================

train_path = PROCESSED_DIR / "train_interactions.parquet"
validation_path = PROCESSED_DIR / "validation_interactions.parquet"
test_path = PROCESSED_DIR / "test_interactions.parquet"

validation_known_path = PROCESSED_DIR / "validation_interactions_known.parquet"
test_known_path = PROCESSED_DIR / "test_interactions_known.parquet"

train_interactions.to_parquet(train_path, index=False)
validation_interactions.to_parquet(validation_path, index=False)
test_interactions.to_parquet(test_path, index=False)

validation_interactions_known.to_parquet(validation_known_path, index=False)
test_interactions_known.to_parquet(test_known_path, index=False)

print("Saved files:")
print(train_path)
print(validation_path)
print(test_path)
print(validation_known_path)
print(test_known_path)

Saved files:
C:\Users\pc\Documents\recommender_project\trial two\data_preprocessed\train_interactions.parquet
C:\Users\pc\Documents\recommender_project\trial two\data_preprocessed\validation_interactions.parquet
C:\Users\pc\Documents\recommender_project\trial two\data_preprocessed\test_interactions.parquet
C:\Users\pc\Documents\recommender_project\trial two\data_preprocessed\validation_interactions_known.parquet
C:\Users\pc\Documents\recommender_project\trial two\data_preprocessed\test_interactions_known.parquet


In [45]:
# =========================
# Step 7.1 - Encode users and items
# =========================

from sklearn.preprocessing import LabelEncoder

user_encoder = LabelEncoder()
item_encoder = LabelEncoder()

# Fit encoders ONLY on train data
train_interactions["user_idx"] = user_encoder.fit_transform(train_interactions["visitorid"])
train_interactions["item_idx"] = item_encoder.fit_transform(train_interactions["itemid"])

# Transform validation/test known data using the same encoders
validation_interactions_known["user_idx"] = user_encoder.transform(
    validation_interactions_known["visitorid"]
)

validation_interactions_known["item_idx"] = item_encoder.transform(
    validation_interactions_known["itemid"]
)

test_interactions_known["user_idx"] = user_encoder.transform(
    test_interactions_known["visitorid"]
)

test_interactions_known["item_idx"] = item_encoder.transform(
    test_interactions_known["itemid"]
)

num_users = len(user_encoder.classes_)
num_items = len(item_encoder.classes_)

print("Encoding complete.")
print("Number of users:", num_users)
print("Number of items:", num_items)

print("\nTrain sample:")
train_interactions.head()

Encoding complete.
Number of users: 22890
Number of items: 18224

Train sample:


,visitorid,itemid,interaction_strength,raw_event_strength,num_events,max_event_weight,first_interaction,last_interaction,is_high_intent,split,user_idx,item_idx
0,64,141657,0.113404,1.0,1,1.0,2015-06-15 21:51:51.942,2015-06-15 21:51:51.942,False,train,0,5543
1,64,233200,0.113491,1.0,1,1.0,2015-06-15 22:40:04.268,2015-06-15 22:40:04.268,False,train,0,9147
2,64,118401,0.113501,1.0,1,1.0,2015-06-15 22:45:31.466,2015-06-15 22:45:31.466,False,train,0,4640
5,155,123027,0.435783,1.0,1,1.0,2015-08-13 04:12:28.331,2015-08-13 04:12:28.331,False,train,1,4812
6,155,134620,0.871564,2.0,2,1.0,2015-08-13 04:11:43.687,2015-08-13 04:12:56.509,False,train,1,5261


In [46]:
# =========================
# Step 7.2 - Check encoded index ranges
# =========================

print("===== USER INDEX CHECK =====")
print("Min user_idx:", train_interactions["user_idx"].min())
print("Max user_idx:", train_interactions["user_idx"].max())
print("Expected max:", num_users - 1)

print("\n===== ITEM INDEX CHECK =====")
print("Min item_idx:", train_interactions["item_idx"].min())
print("Max item_idx:", train_interactions["item_idx"].max())
print("Expected max:", num_items - 1)

print("\nValidation encoded shape:", validation_interactions_known.shape)
print("Test encoded shape:", test_interactions_known.shape)

===== USER INDEX CHECK =====
Min user_idx: 0
Max user_idx: 22889
Expected max: 22889

===== ITEM INDEX CHECK =====
Min item_idx: 0
Max item_idx: 18223
Expected max: 18223

Validation encoded shape: (46825, 12)
Test encoded shape: (46763, 12)


In [47]:
# =========================
# Step 7.3 - Build sparse user-item matrix
# =========================

from scipy.sparse import csr_matrix

train_user_item_matrix = csr_matrix(
    (
        train_interactions["interaction_strength"].astype("float32"),
        (
            train_interactions["user_idx"],
            train_interactions["item_idx"]
        )
    ),
    shape=(num_users, num_items),
    dtype="float32"
)

# Important safety step: combine duplicate entries if any exist
train_user_item_matrix.sum_duplicates()

print("Sparse train user-item matrix created.")
print("Matrix shape:", train_user_item_matrix.shape)
print("Non-zero interactions:", train_user_item_matrix.nnz)

Sparse train user-item matrix created.
Matrix shape: (22890, 18224)
Non-zero interactions: 155049


In [48]:
# =========================
# Step 7.4 - Check sparse matrix sparsity
# =========================

observed_interactions = train_user_item_matrix.nnz
total_possible_interactions = train_user_item_matrix.shape[0] * train_user_item_matrix.shape[1]

density = observed_interactions / total_possible_interactions
sparsity = 1 - density

print("===== TRAIN USER-ITEM MATRIX SPARSITY =====")
print(f"Number of users: {train_user_item_matrix.shape[0]:,}")
print(f"Number of items: {train_user_item_matrix.shape[1]:,}")
print(f"Observed train interactions: {observed_interactions:,}")
print(f"Total possible interactions: {total_possible_interactions:,}")
print(f"Density: {density:.10f}")
print(f"Density percentage: {density * 100:.6f}%")
print(f"Sparsity: {sparsity:.10f}")
print(f"Sparsity percentage: {sparsity * 100:.6f}%")

===== TRAIN USER-ITEM MATRIX SPARSITY =====
Number of users: 22,890
Number of items: 18,224
Observed train interactions: 155,049
Total possible interactions: 417,147,360
Density: 0.0003716888
Density percentage: 0.037169%
Sparsity: 0.9996283112
Sparsity percentage: 99.962831%


In [49]:
# =========================
# Step 7.5 - Save encoders and sparse matrix
# =========================

from scipy.sparse import save_npz
import joblib

matrix_path = PROCESSED_DIR / "train_user_item_matrix.npz"
user_encoder_path = PROCESSED_DIR / "user_encoder.pkl"
item_encoder_path = PROCESSED_DIR / "item_encoder.pkl"

save_npz(matrix_path, train_user_item_matrix)

joblib.dump(user_encoder, user_encoder_path)
joblib.dump(item_encoder, item_encoder_path)

print("Saved train sparse matrix to:")
print(matrix_path)

print("\nSaved user encoder to:")
print(user_encoder_path)

print("\nSaved item encoder to:")
print(item_encoder_path)

Saved train sparse matrix to:
C:\Users\pc\Documents\recommender_project\trial two\data_preprocessed\train_user_item_matrix.npz

Saved user encoder to:
C:\Users\pc\Documents\recommender_project\trial two\data_preprocessed\user_encoder.pkl

Saved item encoder to:
C:\Users\pc\Documents\recommender_project\trial two\data_preprocessed\item_encoder.pkl


In [50]:
# =========================
# Step 7.6 - Save encoded train/validation/test files
# =========================

train_encoded_path = PROCESSED_DIR / "train_interactions_encoded.parquet"
validation_encoded_path = PROCESSED_DIR / "validation_interactions_known_encoded.parquet"
test_encoded_path = PROCESSED_DIR / "test_interactions_known_encoded.parquet"

train_interactions.to_parquet(train_encoded_path, index=False)
validation_interactions_known.to_parquet(validation_encoded_path, index=False)
test_interactions_known.to_parquet(test_encoded_path, index=False)

print("Saved encoded interaction files:")
print(train_encoded_path)
print(validation_encoded_path)
print(test_encoded_path)

Saved encoded interaction files:
C:\Users\pc\Documents\recommender_project\trial two\data_preprocessed\train_interactions_encoded.parquet
C:\Users\pc\Documents\recommender_project\trial two\data_preprocessed\validation_interactions_known_encoded.parquet
C:\Users\pc\Documents\recommender_project\trial two\data_preprocessed\test_interactions_known_encoded.parquet


In [51]:
# =========================
# Step 7.7 - Save user seen items
# =========================

user_seen_items = (
    train_interactions
    .groupby("visitorid")["itemid"]
    .apply(lambda x: list(set(x)))
    .to_dict()
)

user_seen_items_idx = (
    train_interactions
    .groupby("user_idx")["item_idx"]
    .apply(lambda x: list(set(x)))
    .to_dict()
)

joblib.dump(user_seen_items, PROCESSED_DIR / "user_seen_items.pkl")
joblib.dump(user_seen_items_idx, PROCESSED_DIR / "user_seen_items_idx.pkl")

print("Saved user_seen_items.pkl")
print("Saved user_seen_items_idx.pkl")

print("\nExample original user seen items:")
example_user = list(user_seen_items.keys())[0]
print("visitorid:", example_user)
print("seen itemids:", user_seen_items[example_user][:10])

print("\nExample encoded user seen items:")
example_user_idx = list(user_seen_items_idx.keys())[0]
print("user_idx:", example_user_idx)
print("seen item_idx:", user_seen_items_idx[example_user_idx][:10])

Saved user_seen_items.pkl
Saved user_seen_items_idx.pkl

Example original user seen items:
visitorid: 64
seen itemids: [233200, 141657, 118401]

Example encoded user seen items:
user_idx: 0
seen item_idx: [4640, 9147, 5543]


In [52]:
# =========================
# Step 7.8 - Check saved artifacts
# =========================

step_7_files = [
    PROCESSED_DIR / "train_user_item_matrix.npz",
    PROCESSED_DIR / "user_encoder.pkl",
    PROCESSED_DIR / "item_encoder.pkl",
    PROCESSED_DIR / "train_interactions_encoded.parquet",
    PROCESSED_DIR / "validation_interactions_known_encoded.parquet",
    PROCESSED_DIR / "test_interactions_known_encoded.parquet",
    PROCESSED_DIR / "user_seen_items.pkl",
    PROCESSED_DIR / "user_seen_items_idx.pkl"
]

print("===== STEP 7 ARTIFACT CHECK =====")

all_found = True

for file_path in step_7_files:
    if file_path.exists():
        print("FOUND:", file_path.name)
    else:
        print("MISSING:", file_path.name)
        all_found = False

if all_found:
    print("\nStep 7 complete. Sparse matrix and encoders are ready for Milestone 2.")
else:
    print("\nSome files are missing. Check previous cells.")

===== STEP 7 ARTIFACT CHECK =====
FOUND: train_user_item_matrix.npz
FOUND: user_encoder.pkl
FOUND: item_encoder.pkl
FOUND: train_interactions_encoded.parquet
FOUND: validation_interactions_known_encoded.parquet
FOUND: test_interactions_known_encoded.parquet
FOUND: user_seen_items.pkl
FOUND: user_seen_items_idx.pkl

Step 7 complete. Sparse matrix and encoders are ready for Milestone 2.


In [53]:
# =========================
# Step 8.1 - Create ground truth DataFrames
# =========================

# All interactions are considered relevant
validation_ground_truth_all = validation_interactions_known.copy()
test_ground_truth_all = test_interactions_known.copy()

# Only high-intent interactions are considered relevant
# High-intent = addtocart or transaction
validation_ground_truth_high_intent = validation_interactions_known[
    validation_interactions_known["is_high_intent"] == True
].copy()

test_ground_truth_high_intent = test_interactions_known[
    test_interactions_known["is_high_intent"] == True
].copy()

print("===== GROUND TRUTH DATAFRAMES =====")
print("Validation all:", validation_ground_truth_all.shape)
print("Validation high-intent:", validation_ground_truth_high_intent.shape)
print("Test all:", test_ground_truth_all.shape)
print("Test high-intent:", test_ground_truth_high_intent.shape)

===== GROUND TRUTH DATAFRAMES =====
Validation all: (46825, 12)
Validation high-intent: (3852, 12)
Test all: (46763, 12)
Test high-intent: (5267, 12)


In [54]:
# =========================
# Step 8.2 - Check users with high-intent ground truth
# =========================

print("===== USERS WITH GROUND TRUTH =====")

print("Validation all users:", validation_ground_truth_all["visitorid"].nunique())
print("Validation high-intent users:", validation_ground_truth_high_intent["visitorid"].nunique())

print("\nTest all users:", test_ground_truth_all["visitorid"].nunique())
print("Test high-intent users:", test_ground_truth_high_intent["visitorid"].nunique())

print("\nHigh-intent validation percentage:")
print(
    validation_ground_truth_high_intent["visitorid"].nunique()
    / validation_ground_truth_all["visitorid"].nunique()
    * 100
)

print("\nHigh-intent test percentage:")
print(
    test_ground_truth_high_intent["visitorid"].nunique()
    / test_ground_truth_all["visitorid"].nunique()
    * 100
)

===== USERS WITH GROUND TRUTH =====
Validation all users: 22869
Validation high-intent users: 2260

Test all users: 22860
Test high-intent users: 3178

High-intent validation percentage:
9.882373518737154

High-intent test percentage:
13.902012248468942


In [55]:
# =========================
# Step 8.3 - Convert ground truth to dictionaries
# =========================

def build_ground_truth_dict(df, user_col="user_idx", item_col="item_idx"):
    """
    Convert ground truth DataFrame into dictionary:
    user_idx -> list of relevant item_idx values
    """
    ground_truth = (
        df
        .groupby(user_col)[item_col]
        .apply(lambda x: list(set(x)))
        .to_dict()
    )
    
    return ground_truth


validation_gt_all_dict = build_ground_truth_dict(validation_ground_truth_all)
test_gt_all_dict = build_ground_truth_dict(test_ground_truth_all)

validation_gt_high_intent_dict = build_ground_truth_dict(validation_ground_truth_high_intent)
test_gt_high_intent_dict = build_ground_truth_dict(test_ground_truth_high_intent)

print("===== GROUND TRUTH DICTIONARIES =====")
print("Validation all users:", len(validation_gt_all_dict))
print("Validation high-intent users:", len(validation_gt_high_intent_dict))
print("Test all users:", len(test_gt_all_dict))
print("Test high-intent users:", len(test_gt_high_intent_dict))

# Show one example
example_user_idx = list(test_gt_all_dict.keys())[0]
print("\nExample user_idx:", example_user_idx)
print("Relevant test item_idx:", test_gt_all_dict[example_user_idx][:10])

===== GROUND TRUTH DICTIONARIES =====
Validation all users: 22869
Validation high-intent users: 2260
Test all users: 22860
Test high-intent users: 3178

Example user_idx: 0
Relevant test item_idx: [6273]


In [56]:
# =========================
# Step 8.4 - Save ground truth files
# =========================

# Save DataFrames
validation_ground_truth_all.to_parquet(
    PROCESSED_DIR / "validation_ground_truth_all.parquet",
    index=False
)

test_ground_truth_all.to_parquet(
    PROCESSED_DIR / "test_ground_truth_all.parquet",
    index=False
)

validation_ground_truth_high_intent.to_parquet(
    PROCESSED_DIR / "validation_ground_truth_high_intent.parquet",
    index=False
)

test_ground_truth_high_intent.to_parquet(
    PROCESSED_DIR / "test_ground_truth_high_intent.parquet",
    index=False
)

# Save dictionaries
joblib.dump(
    validation_gt_all_dict,
    PROCESSED_DIR / "validation_gt_all_dict.pkl"
)

joblib.dump(
    test_gt_all_dict,
    PROCESSED_DIR / "test_gt_all_dict.pkl"
)

joblib.dump(
    validation_gt_high_intent_dict,
    PROCESSED_DIR / "validation_gt_high_intent_dict.pkl"
)

joblib.dump(
    test_gt_high_intent_dict,
    PROCESSED_DIR / "test_gt_high_intent_dict.pkl"
)

print("Ground truth files saved successfully.")

Ground truth files saved successfully.


In [57]:
# =========================
# Step 8.5 - Create evaluation user lists
# =========================

validation_users_all = sorted(list(validation_gt_all_dict.keys()))
test_users_all = sorted(list(test_gt_all_dict.keys()))

validation_users_high_intent = sorted(list(validation_gt_high_intent_dict.keys()))
test_users_high_intent = sorted(list(test_gt_high_intent_dict.keys()))

joblib.dump(validation_users_all, PROCESSED_DIR / "validation_users_all.pkl")
joblib.dump(test_users_all, PROCESSED_DIR / "test_users_all.pkl")

joblib.dump(validation_users_high_intent, PROCESSED_DIR / "validation_users_high_intent.pkl")
joblib.dump(test_users_high_intent, PROCESSED_DIR / "test_users_high_intent.pkl")

print("Evaluation user lists saved.")

print("\nValidation all users:", len(validation_users_all))
print("Test all users:", len(test_users_all))

print("\nValidation high-intent users:", len(validation_users_high_intent))
print("Test high-intent users:", len(test_users_high_intent))

Evaluation user lists saved.

Validation all users: 22869
Test all users: 22860

Validation high-intent users: 2260
Test high-intent users: 3178


In [58]:
# =========================
# Step 8.6 - Ground truth artifact check
# =========================

step_8_files = [
    PROCESSED_DIR / "validation_ground_truth_all.parquet",
    PROCESSED_DIR / "test_ground_truth_all.parquet",
    PROCESSED_DIR / "validation_ground_truth_high_intent.parquet",
    PROCESSED_DIR / "test_ground_truth_high_intent.parquet",
    
    PROCESSED_DIR / "validation_gt_all_dict.pkl",
    PROCESSED_DIR / "test_gt_all_dict.pkl",
    PROCESSED_DIR / "validation_gt_high_intent_dict.pkl",
    PROCESSED_DIR / "test_gt_high_intent_dict.pkl",
    
    PROCESSED_DIR / "validation_users_all.pkl",
    PROCESSED_DIR / "test_users_all.pkl",
    PROCESSED_DIR / "validation_users_high_intent.pkl",
    PROCESSED_DIR / "test_users_high_intent.pkl"
]

print("===== STEP 8 ARTIFACT CHECK =====")

all_found = True

for file_path in step_8_files:
    if file_path.exists():
        print("FOUND:", file_path.name)
    else:
        print("MISSING:", file_path.name)
        all_found = False

if all_found:
    print("\nStep 8 complete. Ground truth files are ready for Milestone 2 evaluation.")
else:
    print("\nSome files are missing. Check previous cells.")

===== STEP 8 ARTIFACT CHECK =====
FOUND: validation_ground_truth_all.parquet
FOUND: test_ground_truth_all.parquet
FOUND: validation_ground_truth_high_intent.parquet
FOUND: test_ground_truth_high_intent.parquet
FOUND: validation_gt_all_dict.pkl
FOUND: test_gt_all_dict.pkl
FOUND: validation_gt_high_intent_dict.pkl
FOUND: test_gt_high_intent_dict.pkl
FOUND: validation_users_all.pkl
FOUND: test_users_all.pkl
FOUND: validation_users_high_intent.pkl
FOUND: test_users_high_intent.pkl

Step 8 complete. Ground truth files are ready for Milestone 2 evaluation.


In [59]:

# Step 9.1 - Get model item IDs


model_item_ids = set(item_encoder.classes_.tolist())

print("Number of model items:", len(model_item_ids))
print("Example item IDs:", list(model_item_ids)[:10])

Number of model items: 18224
Example item IDs: [65540, 98308, 229382, 229384, 229388, 15, 98320, 163856, 262159, 196629]


In [60]:
# =========================
# Step 9.2 - Define chunked loader for item properties
# =========================

def load_item_properties_for_model_items(file_path, model_item_ids, chunksize=500_000):
    """
    Load only item property rows for items that exist in our model.

    This avoids loading unnecessary metadata for items we will not recommend.
    """
    
    selected_chunks = []
    total_rows_read = 0
    total_rows_kept = 0
    
    print(f"Reading: {file_path.name}")
    
    for chunk_number, chunk in enumerate(
        pd.read_csv(
            file_path,
            chunksize=chunksize,
            usecols=["timestamp", "itemid", "property", "value"],
            dtype={
                "timestamp": "int64",
                "itemid": "int64",
                "property": "string",
                "value": "string"
            }
        ),
        start=1
    ):
        total_rows_read += len(chunk)
        
        chunk_filtered = chunk[chunk["itemid"].isin(model_item_ids)].copy()
        
        total_rows_kept += len(chunk_filtered)
        
        if len(chunk_filtered) > 0:
            selected_chunks.append(chunk_filtered)
        
        print(
            f"Chunk {chunk_number}: read {len(chunk):,} rows, "
            f"kept {len(chunk_filtered):,} rows"
        )
    
    if len(selected_chunks) > 0:
        result = pd.concat(selected_chunks, ignore_index=True)
    else:
        result = pd.DataFrame(columns=["timestamp", "itemid", "property", "value"])
    
    print("-" * 60)
    print(f"Finished: {file_path.name}")
    print(f"Total rows read: {total_rows_read:,}")
    print(f"Total rows kept: {total_rows_kept:,}")
    print(f"Result shape: {result.shape}")
    
    return result

In [61]:
# =========================
# Step 9.3 - Load relevant item properties from both files
# =========================

item_props_part1_model = load_item_properties_for_model_items(
    item_properties_part1_path,
    model_item_ids,
    chunksize=500_000
)

item_props_part2_model = load_item_properties_for_model_items(
    item_properties_part2_path,
    model_item_ids,
    chunksize=500_000
)

item_properties_model = pd.concat(
    [item_props_part1_model, item_props_part2_model],
    ignore_index=True
)

# Free memory
del item_props_part1_model
del item_props_part2_model

print("\nCombined model item properties shape:", item_properties_model.shape)
item_properties_model.head()

Reading: item_properties_part1.csv
Chunk 1: read 500,000 rows, kept 32,351 rows
Chunk 2: read 500,000 rows, kept 32,104 rows
Chunk 3: read 500,000 rows, kept 32,143 rows
Chunk 4: read 500,000 rows, kept 32,123 rows
Chunk 5: read 500,000 rows, kept 32,307 rows
Chunk 6: read 500,000 rows, kept 31,861 rows
Chunk 7: read 500,000 rows, kept 32,448 rows
Chunk 8: read 500,000 rows, kept 32,225 rows
Chunk 9: read 500,000 rows, kept 32,684 rows
Chunk 10: read 500,000 rows, kept 32,376 rows
Chunk 11: read 500,000 rows, kept 32,533 rows
Chunk 12: read 500,000 rows, kept 32,722 rows
Chunk 13: read 500,000 rows, kept 32,432 rows
Chunk 14: read 500,000 rows, kept 32,482 rows
Chunk 15: read 500,000 rows, kept 32,484 rows
Chunk 16: read 500,000 rows, kept 32,442 rows
Chunk 17: read 500,000 rows, kept 32,645 rows
Chunk 18: read 500,000 rows, kept 32,589 rows
Chunk 19: read 500,000 rows, kept 32,642 rows
Chunk 20: read 500,000 rows, kept 32,464 rows
Chunk 21: read 500,000 rows, kept 32,374 rows
Chunk 22

,timestamp,itemid,property,value
0,1439694000000,48696,566,n480.000 639502 189174
1,1434250800000,269797,159,519769
2,1435460400000,334428,216,637368 190776
3,1435460400000,279407,558,1214748 1186610
4,1434250800000,184487,1079,769062


In [62]:
# =========================
# Step 9.4 - Convert item property timestamps
# =========================

item_properties_model["timestamp"] = pd.to_datetime(
    item_properties_model["timestamp"],
    unit="ms"
)

print("Timestamp converted.")
print("Date range:")
print("Start:", item_properties_model["timestamp"].min())
print("End:", item_properties_model["timestamp"].max())

print("\nShape:", item_properties_model.shape)
item_properties_model.head()

Timestamp converted.
Date range:
Start: 2015-05-10 03:00:00
End: 2015-09-13 03:00:00

Shape: (1311358, 4)


,timestamp,itemid,property,value
0,2015-08-16 03:00:00,48696,566,n480.000 639502 189174
1,2015-06-14 03:00:00,269797,159,519769
2,2015-06-28 03:00:00,334428,216,637368 190776
3,2015-06-28 03:00:00,279407,558,1214748 1186610
4,2015-06-14 03:00:00,184487,1079,769062


In [63]:
# =========================
# Step 9.5 - Item properties overview
# =========================

print("Item properties shape:", item_properties_model.shape)
print("Unique items with metadata:", item_properties_model["itemid"].nunique())
print("Unique properties:", item_properties_model["property"].nunique())

print("\nTop 20 properties:")
print(item_properties_model["property"].value_counts().head(20))

print("\nMissing values:")
print(item_properties_model.isna().sum())

Item properties shape: (1311358, 4)
Unique items with metadata: 17542
Unique properties: 939

Top 20 properties:
property
790           282954
available     206776
888           175842
categoryid     36932
400            33347
283            25474
776            24046
364            22154
202            19866
678            19391
6              19102
839            18996
917            17932
764            17542
112            17542
159            17542
227            16118
1036           11627
698            11614
928             9191
Name: count, dtype: Int64

Missing values:
timestamp    0
itemid       0
property     0
value        0
dtype: int64


In [64]:
# =========================
# Step 9.6 - Save filtered item properties
# =========================

filtered_item_properties_path = PROCESSED_DIR / "item_properties_model_filtered.parquet"

item_properties_model.to_parquet(filtered_item_properties_path, index=False)

print("Saved filtered item properties to:")
print(filtered_item_properties_path)

Saved filtered item properties to:
C:\Users\pc\Documents\recommender_project\trial two\data_preprocessed\item_properties_model_filtered.parquet


In [65]:
# =========================
# Step 9.7 - Keep latest value per item-property
# =========================

item_properties_model = item_properties_model.sort_values(
    ["itemid", "property", "timestamp"]
)

latest_item_properties = (
    item_properties_model
    .groupby(["itemid", "property"], as_index=False)
    .tail(1)
    .reset_index(drop=True)
)

print("Latest item-property table created.")
print("Shape:", latest_item_properties.shape)
print("Unique items:", latest_item_properties["itemid"].nunique())
print("Unique properties:", latest_item_properties["property"].nunique())

latest_item_properties.head()

Latest item-property table created.
Shape: (522721, 4)
Unique items: 17542
Unique properties: 939


,timestamp,itemid,property,value
0,2015-06-28 03:00:00,15,112,679677
1,2015-06-07 03:00:00,15,159,519769
2,2015-05-31 03:00:00,15,202,789221
3,2015-06-07 03:00:00,15,227,433564
4,2015-05-10 03:00:00,15,283,433564 245772 789221 809278 245772 1213953 429...


In [66]:
# =========================
# Step 9.8 - Extract categoryid and available features
# =========================

# Category feature
category_features = latest_item_properties[
    latest_item_properties["property"] == "categoryid"
][["itemid", "value"]].copy()

category_features = category_features.rename(columns={"value": "categoryid"})

# Availability feature
availability_features = latest_item_properties[
    latest_item_properties["property"] == "available"
][["itemid", "value"]].copy()

availability_features = availability_features.rename(columns={"value": "available"})

print("Category features shape:", category_features.shape)
print("Availability features shape:", availability_features.shape)

print("\nCategory sample:")
display(category_features.head())

print("\nAvailability sample:")
display(availability_features.head())

Category features shape: (17542, 2)
Availability features shape: (17542, 2)

Category sample:


,itemid,categoryid
20,15,722
45,25,72
66,26,1503
100,42,84
127,102,242



Availability sample:


,itemid,available
19,15,0
44,25,1
65,26,0
99,42,1
126,102,1


In [67]:
# =========================
# Step 9.9 - Build property text per item
# =========================

latest_item_properties["property_value_text"] = (
    latest_item_properties["property"].astype(str)
    + "_"
    + latest_item_properties["value"].astype(str)
)

item_property_text = (
    latest_item_properties
    .groupby("itemid")["property_value_text"]
    .apply(lambda values: " ".join(values))
    .reset_index()
    .rename(columns={"property_value_text": "property_text"})
)

print("Item property text created.")
print("Shape:", item_property_text.shape)

item_property_text.head()

Item property text created.
Shape: (17542, 2)


,itemid,property_text
0,15,112_679677 159_519769 202_789221 227_433564 28...
1,25,1036_1154859 105_769062 1077_140221 112_679677...
2,26,112_679677 159_519769 202_234027 309875 283_66...
3,42,1036_726612 1052_1116693 1066_n68.400 424566 1...
4,102,112_679677 159_519769 19_1297729 n36.000 35072...


In [68]:
# =========================
# Step 9.10 - Build final item features table
# =========================

item_features = pd.DataFrame({
    "itemid": sorted(model_item_ids)
})

item_features = item_features.merge(
    category_features,
    on="itemid",
    how="left"
)

item_features = item_features.merge(
    availability_features,
    on="itemid",
    how="left"
)

item_features = item_features.merge(
    item_property_text,
    on="itemid",
    how="left"
)

# Fill missing metadata
item_features["categoryid"] = item_features["categoryid"].fillna("unknown")
item_features["available"] = item_features["available"].fillna("unknown")
item_features["property_text"] = item_features["property_text"].fillna("")

# Add item_idx so it aligns with the recommender matrix
item_features["item_idx"] = item_encoder.transform(item_features["itemid"])

# Sort by item_idx to align rows with train_user_item_matrix columns
item_features = item_features.sort_values("item_idx").reset_index(drop=True)

print("Final item features created.")
print("Shape:", item_features.shape)

item_features.head()

Final item features created.
Shape: (18224, 5)


,itemid,categoryid,available,property_text,item_idx
0,15,722,0,112_679677 159_519769 202_789221 227_433564 28...,0
1,25,72,1,1036_1154859 105_769062 1077_140221 112_679677...,1
2,26,1503,0,112_679677 159_519769 202_234027 309875 283_66...,2
3,42,84,1,1036_726612 1052_1116693 1066_n68.400 424566 1...,3
4,102,242,1,112_679677 159_519769 19_1297729 n36.000 35072...,4


In [69]:
# =========================
# Step 9.11 - Check metadata coverage
# =========================

total_model_items = len(item_features)

items_with_category = (item_features["categoryid"] != "unknown").sum()
items_with_availability = (item_features["available"] != "unknown").sum()
items_with_property_text = (item_features["property_text"].str.len() > 0).sum()

print("===== ITEM METADATA COVERAGE =====")
print(f"Total model items: {total_model_items:,}")

print(f"Items with category: {items_with_category:,}")
print(f"Category coverage: {items_with_category / total_model_items * 100:.2f}%")

print(f"\nItems with availability: {items_with_availability:,}")
print(f"Availability coverage: {items_with_availability / total_model_items * 100:.2f}%")

print(f"\nItems with property text: {items_with_property_text:,}")
print(f"Property text coverage: {items_with_property_text / total_model_items * 100:.2f}%")

print("\nTop categories:")
print(item_features["categoryid"].value_counts().head(10))

print("\nAvailability distribution:")
print(item_features["available"].value_counts())

===== ITEM METADATA COVERAGE =====
Total model items: 18,224
Items with category: 17,542
Category coverage: 96.26%

Items with availability: 17,542
Availability coverage: 96.26%

Items with property text: 17,542
Property text coverage: 96.26%

Top categories:
categoryid
unknown    682
1051       559
959        379
1135       336
1483       318
491        301
1279       291
342        285
84         275
589        275
Name: count, dtype: Int64

Availability distribution:
available
1          10697
0           6845
unknown      682
Name: count, dtype: Int64


In [70]:
# =========================
# Step 9.12 - Save item features
# =========================

item_features_path = PROCESSED_DIR / "item_features.parquet"

item_features.to_parquet(item_features_path, index=False)

print("Saved item features to:")
print(item_features_path)

print("\nItem features shape:", item_features.shape)
print("Columns:")
print(item_features.columns.tolist())

Saved item features to:
C:\Users\pc\Documents\recommender_project\trial two\data_preprocessed\item_features.parquet

Item features shape: (18224, 5)
Columns:
['itemid', 'categoryid', 'available', 'property_text', 'item_idx']


In [71]:
# =========================
# Step 9.13 - Build TF-IDF matrix for item text
# =========================

from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import save_npz
import joblib

tfidf_vectorizer = TfidfVectorizer(
    max_features=300,
    min_df=2,
    max_df=0.95
)

item_tfidf_matrix = tfidf_vectorizer.fit_transform(
    item_features["property_text"]
)

print("TF-IDF matrix created.")
print("Shape:", item_tfidf_matrix.shape)
print("Number of TF-IDF features:", len(tfidf_vectorizer.get_feature_names_out()))

TF-IDF matrix created.
Shape: (18224, 300)
Number of TF-IDF features: 300


In [72]:
# =========================
# Step 9.14 - Save TF-IDF matrix and vectorizer
# =========================

item_tfidf_matrix_path = PROCESSED_DIR / "item_tfidf_matrix.npz"
tfidf_vectorizer_path = PROCESSED_DIR / "tfidf_vectorizer.pkl"

save_npz(item_tfidf_matrix_path, item_tfidf_matrix)
joblib.dump(tfidf_vectorizer, tfidf_vectorizer_path)

print("Saved item TF-IDF matrix to:")
print(item_tfidf_matrix_path)

print("\nSaved TF-IDF vectorizer to:")
print(tfidf_vectorizer_path)

Saved item TF-IDF matrix to:
C:\Users\pc\Documents\recommender_project\trial two\data_preprocessed\item_tfidf_matrix.npz

Saved TF-IDF vectorizer to:
C:\Users\pc\Documents\recommender_project\trial two\data_preprocessed\tfidf_vectorizer.pkl


In [74]:
split_summary = pd.DataFrame({
    "split": ["train", "validation_known", "test_known"],
    "rows": [
        len(train_interactions),
        len(validation_interactions_known),
        len(test_interactions_known)
    ],
    "unique_users": [
        train_interactions["visitorid"].nunique(),
        validation_interactions_known["visitorid"].nunique(),
        test_interactions_known["visitorid"].nunique()
    ],
    "unique_items": [
        train_interactions["itemid"].nunique(),
        validation_interactions_known["itemid"].nunique(),
        test_interactions_known["itemid"].nunique()
    ],
    "high_intent_rows": [
        train_interactions["is_high_intent"].sum(),
        validation_interactions_known["is_high_intent"].sum(),
        test_interactions_known["is_high_intent"].sum()
    ]
})

display(split_summary)
split_summary.to_csv(PROCESSED_DIR / "split_summary.csv", index=False)

,split,rows,unique_users,unique_items,high_intent_rows
0,train,155049,22890,18224,11350
1,validation_known,46825,22869,15270,3852
2,test_known,46763,22860,15152,5267


In [75]:
raw_num_users = interactions["visitorid"].nunique()
raw_num_items = interactions["itemid"].nunique()
raw_observed_pairs = len(interactions)
raw_total_pairs = raw_num_users * raw_num_items
raw_density = raw_observed_pairs / raw_total_pairs
raw_sparsity = 1 - raw_density

filtered_num_users = interactions_filtered["visitorid"].nunique()
filtered_num_items = interactions_filtered["itemid"].nunique()
filtered_observed_pairs = len(interactions_filtered)
filtered_total_pairs = filtered_num_users * filtered_num_items
filtered_density = filtered_observed_pairs / filtered_total_pairs
filtered_sparsity = 1 - filtered_density

train_observed_pairs = train_user_item_matrix.nnz
train_total_pairs = train_user_item_matrix.shape[0] * train_user_item_matrix.shape[1]
train_density = train_observed_pairs / train_total_pairs
train_sparsity = 1 - train_density

sparsity_summary = pd.DataFrame({
    "stage": ["before_filtering", "after_filtering", "train_matrix"],
    "num_users": [raw_num_users, filtered_num_users, train_user_item_matrix.shape[0]],
    "num_items": [raw_num_items, filtered_num_items, train_user_item_matrix.shape[1]],
    "observed_pairs": [raw_observed_pairs, filtered_observed_pairs, train_observed_pairs],
    "total_possible_pairs": [raw_total_pairs, filtered_total_pairs, train_total_pairs],
    "density_percentage": [raw_density * 100, filtered_density * 100, train_density * 100],
    "sparsity_percentage": [raw_sparsity * 100, filtered_sparsity * 100, train_sparsity * 100]
})

display(sparsity_summary)
sparsity_summary.to_csv(PROCESSED_DIR / "sparsity_summary.csv", index=False)

,stage,num_users,num_items,observed_pairs,total_possible_pairs,density_percentage,sparsity_percentage
0,before_filtering,1407580,235061,2145179,330867162380,0.000648,99.999352
1,after_filtering,22890,18269,248891,418177410,0.059518,99.940482
2,train_matrix,22890,18224,155049,417147360,0.037169,99.962831


In [76]:
total_model_items = len(item_features)

items_with_category = (item_features["categoryid"] != "unknown").sum()
items_with_availability = (item_features["available"] != "unknown").sum()
items_with_property_text = (item_features["property_text"].str.len() > 0).sum()

metadata_summary = {
    "total_model_items": int(total_model_items),
    "items_with_category": int(items_with_category),
    "category_coverage_percentage": float(items_with_category / total_model_items * 100),
    "items_with_availability": int(items_with_availability),
    "availability_coverage_percentage": float(items_with_availability / total_model_items * 100),
    "items_with_property_text": int(items_with_property_text),
    "property_text_coverage_percentage": float(items_with_property_text / total_model_items * 100),
    "tfidf_matrix_shape": list(item_tfidf_matrix.shape)
}

metadata_summary

{'total_model_items': 18224,
 'items_with_category': 17542,
 'category_coverage_percentage': 96.25768217734854,
 'items_with_availability': 17542,
 'availability_coverage_percentage': 96.25768217734854,
 'items_with_property_text': 17542,
 'property_text_coverage_percentage': 96.25768217734854,
 'tfidf_matrix_shape': [18224, 300]}

In [77]:
required_milestone_1_files = [
    PROCESSED_DIR / "events_clean.parquet",
    PROCESSED_DIR / "events_weighted.parquet",
    PROCESSED_DIR / "interactions_aggregated.parquet",
    PROCESSED_DIR / "interactions_filtered.parquet",

    PROCESSED_DIR / "train_interactions.parquet",
    PROCESSED_DIR / "validation_interactions.parquet",
    PROCESSED_DIR / "test_interactions.parquet",
    PROCESSED_DIR / "validation_interactions_known.parquet",
    PROCESSED_DIR / "test_interactions_known.parquet",

    PROCESSED_DIR / "train_user_item_matrix.npz",
    PROCESSED_DIR / "user_encoder.pkl",
    PROCESSED_DIR / "item_encoder.pkl",

    PROCESSED_DIR / "train_interactions_encoded.parquet",
    PROCESSED_DIR / "validation_interactions_known_encoded.parquet",
    PROCESSED_DIR / "test_interactions_known_encoded.parquet",

    PROCESSED_DIR / "user_seen_items.pkl",
    PROCESSED_DIR / "user_seen_items_idx.pkl",

    PROCESSED_DIR / "validation_ground_truth_all.parquet",
    PROCESSED_DIR / "test_ground_truth_all.parquet",
    PROCESSED_DIR / "validation_ground_truth_high_intent.parquet",
    PROCESSED_DIR / "test_ground_truth_high_intent.parquet",

    PROCESSED_DIR / "validation_gt_all_dict.pkl",
    PROCESSED_DIR / "test_gt_all_dict.pkl",
    PROCESSED_DIR / "validation_gt_high_intent_dict.pkl",
    PROCESSED_DIR / "test_gt_high_intent_dict.pkl",

    PROCESSED_DIR / "item_properties_model_filtered.parquet",
    PROCESSED_DIR / "item_features.parquet",
    PROCESSED_DIR / "item_tfidf_matrix.npz",
    PROCESSED_DIR / "tfidf_vectorizer.pkl",

    PROCESSED_DIR / "split_summary.csv",
    PROCESSED_DIR / "sparsity_summary.csv"
]

print("===== FINAL MILESTONE 1 ARTIFACT CHECK =====")

all_found = True

for file_path in required_milestone_1_files:
    if file_path.exists():
        print("FOUND:", file_path.name)
    else:
        print("MISSING:", file_path.name)
        all_found = False

if all_found:
    print("\nMilestone 1 core artifacts are complete.")
else:
    print("\nSome files are missing.")

===== FINAL MILESTONE 1 ARTIFACT CHECK =====
FOUND: events_clean.parquet
FOUND: events_weighted.parquet
FOUND: interactions_aggregated.parquet
FOUND: interactions_filtered.parquet
FOUND: train_interactions.parquet
FOUND: validation_interactions.parquet
FOUND: test_interactions.parquet
FOUND: validation_interactions_known.parquet
FOUND: test_interactions_known.parquet
FOUND: train_user_item_matrix.npz
FOUND: user_encoder.pkl
FOUND: item_encoder.pkl
FOUND: train_interactions_encoded.parquet
FOUND: validation_interactions_known_encoded.parquet
FOUND: test_interactions_known_encoded.parquet
FOUND: user_seen_items.pkl
FOUND: user_seen_items_idx.pkl
FOUND: validation_ground_truth_all.parquet
FOUND: test_ground_truth_all.parquet
FOUND: validation_ground_truth_high_intent.parquet
FOUND: test_ground_truth_high_intent.parquet
FOUND: validation_gt_all_dict.pkl
FOUND: test_gt_all_dict.pkl
FOUND: validation_gt_high_intent_dict.pkl
FOUND: test_gt_high_intent_dict.pkl
FOUND: item_properties_model_fil

In [78]:
# =========================
# Save metadata summary
# =========================

with open(PROCESSED_DIR / "metadata_summary.json", "w") as f:
    json.dump(metadata_summary, f, indent=4)

print("Saved metadata_summary.json")

Saved metadata_summary.json


In [79]:
# =========================
# Save final Milestone 1 summary
# =========================

milestone_1_summary = {
    "raw_events_rows": int(len(events)),
    "clean_events_rows": int(len(events_clean)),
    "aggregated_user_item_pairs": int(len(interactions)),
    "filtered_user_item_pairs": int(len(interactions_filtered)),

    "raw_unique_users": int(events["visitorid"].nunique()),
    "raw_unique_items": int(events["itemid"].nunique()),

    "filtered_unique_users": int(interactions_filtered["visitorid"].nunique()),
    "filtered_unique_items": int(interactions_filtered["itemid"].nunique()),

    "train_rows": int(len(train_interactions)),
    "validation_known_rows": int(len(validation_interactions_known)),
    "test_known_rows": int(len(test_interactions_known)),

    "train_matrix_shape": [
        int(train_user_item_matrix.shape[0]),
        int(train_user_item_matrix.shape[1])
    ],
    "train_matrix_nonzero": int(train_user_item_matrix.nnz),
    "train_matrix_sparsity_percentage": float(
        100 * (1 - train_user_item_matrix.nnz / (
            train_user_item_matrix.shape[0] * train_user_item_matrix.shape[1]
        ))
    ),

    "event_weights": {
        "view": 1.0,
        "addtocart": 3.0,
        "transaction": 10.0
    },

    "recency_weighting": {
        "used": True,
        "half_life_days": 30
    },

    "filtering": {
        "min_user_interactions": 5,
        "min_item_interactions": 5
    },

    "metadata_summary": metadata_summary
}

with open(PROCESSED_DIR / "milestone_1_summary.json", "w") as f:
    json.dump(milestone_1_summary, f, indent=4)

print("Saved milestone_1_summary.json")

Saved milestone_1_summary.json


In [80]:
# =========================
# Check 1 - Data split leakage checks
# =========================

def check_split_leakage(train_df, validation_df, test_df):
    """
    Check if there is leakage between train, validation, and test splits.
    """

    print("=" * 60)
    print("CHECK 1: SPLIT LEAKAGE")
    print("=" * 60)

    # 1. Check exact user-item overlap
    train_pairs = set(zip(train_df["user_idx"], train_df["item_idx"]))
    validation_pairs = set(zip(validation_df["user_idx"], validation_df["item_idx"]))
    test_pairs = set(zip(test_df["user_idx"], test_df["item_idx"]))

    train_validation_overlap = train_pairs & validation_pairs
    train_test_overlap = train_pairs & test_pairs
    validation_test_overlap = validation_pairs & test_pairs

    print("Train/Validation user-item overlap:", len(train_validation_overlap))
    print("Train/Test user-item overlap:", len(train_test_overlap))
    print("Validation/Test user-item overlap:", len(validation_test_overlap))

    # 2. Check cold-start users/items
    train_users = set(train_df["user_idx"])
    train_items = set(train_df["item_idx"])

    validation_users = set(validation_df["user_idx"])
    validation_items = set(validation_df["item_idx"])

    test_users = set(test_df["user_idx"])
    test_items = set(test_df["item_idx"])

    print("\nCold-start users in validation:", len(validation_users - train_users))
    print("Cold-start items in validation:", len(validation_items - train_items))

    print("Cold-start users in test:", len(test_users - train_users))
    print("Cold-start items in test:", len(test_items - train_items))

    # 3. Per-user chronological leakage
    train_max_time = (
        train_df
        .groupby("user_idx")["last_interaction"]
        .max()
        .rename("train_max_time")
    )

    validation_min_time = (
        validation_df
        .groupby("user_idx")["last_interaction"]
        .min()
        .rename("validation_min_time")
    )

    test_min_time = (
        test_df
        .groupby("user_idx")["last_interaction"]
        .min()
        .rename("test_min_time")
    )

    validation_time_check = pd.concat(
        [train_max_time, validation_min_time],
        axis=1
    ).dropna()

    test_time_check = pd.concat(
        [train_max_time, test_min_time],
        axis=1
    ).dropna()

    validation_time_leakage = (
        validation_time_check["validation_min_time"]
        < validation_time_check["train_max_time"]
    ).sum()

    test_time_leakage = (
        test_time_check["test_min_time"]
        < test_time_check["train_max_time"]
    ).sum()

    print("\nValidation chronological leakage users:", validation_time_leakage)
    print("Test chronological leakage users:", test_time_leakage)

    # Final result
    total_issues = (
        len(train_validation_overlap)
        + len(train_test_overlap)
        + len(validation_test_overlap)
        + len(validation_users - train_users)
        + len(validation_items - train_items)
        + len(test_users - train_users)
        + len(test_items - train_items)
        + validation_time_leakage
        + test_time_leakage
    )

    print("\n" + "-" * 60)

    if total_issues == 0:
        print("PASS: No obvious split leakage detected.")
    else:
        print("WARNING: Potential leakage/issues detected.")

    return total_issues


split_leakage_issues = check_split_leakage(
    train_interactions,
    validation_interactions_known,
    test_interactions_known
)

CHECK 1: SPLIT LEAKAGE
Train/Validation user-item overlap: 0
Train/Test user-item overlap: 0
Validation/Test user-item overlap: 0

Cold-start users in validation: 0
Cold-start items in validation: 0
Cold-start users in test: 0
Cold-start items in test: 0

Validation chronological leakage users: 0
Test chronological leakage users: 0

------------------------------------------------------------
PASS: No obvious split leakage detected.
